# Yahoo Finance Exploration for Bronze Layer

This notebook helps us understand which Yahoo Finance datasets are useful for the project and how they should land in the bronze layer.

Project goal for this notebook:
- explore daily stock price history
- inspect corporate actions such as dividends and splits
- inspect company metadata useful for dimensions later
- design the raw bronze file layout we will use in S3 later


## Step 1. Choose a small sample universe

For the first exploration, keep the scope small and representative:
- US equities: `AAPL`, `MSFT`, `NVDA`
- Brazil equities: `PETR4.SA`, `VALE3.SA`
- ETF example: `SPY`
- Crypto can be explored later in a separate notebook

This is enough to test different geographies and ticker formats without making the notebook noisy.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd
import yfinance as yf

In [ ]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

sample_tickers = ["AAPL", "MSFT", "NVDA", "PETR4.SA", "VALE3.SA", "SPY"]
analysis_end_date = pd.Timestamp.today().normalize()
analysis_start_date = analysis_end_date - pd.Timedelta(days=180)

print("Sample tickers:", sample_tickers)
print("Analysis window:", analysis_start_date.date(), "to", analysis_end_date.date())

## Step 2. Explore daily market prices

The first bronze dataset should be daily OHLCV prices because it supports almost every downstream use case:
- trend analysis
- volatility metrics
- dashboard time series
- joins with company and sector metadata later

We will start by checking what `yfinance` returns for one ticker.

In [ ]:
ticker = yf.Ticker("AAPL")

aapl_history = ticker.history(
    start=analysis_start_date.strftime("%Y-%m-%d"),
    end=analysis_end_date.strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=False,
    actions=True,
)

aapl_history.head()

In [ ]:
print("Shape:", aapl_history.shape)
print("Index type:", type(aapl_history.index))
print("Columns:", list(aapl_history.columns))
print("Null counts:")
aapl_history.isna().sum()

## Step 3. Download multiple tickers

For the pipeline, we will ingest multiple tickers per run. Here we test the multi-ticker output and then reshape it into a normalized format.

In [ ]:
raw_multi_prices = yf.download(
    tickers=sample_tickers,
    start=analysis_start_date.strftime("%Y-%m-%d"),
    end=analysis_end_date.strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=False,
    actions=True,
    group_by="ticker",
    progress=False,
)

raw_multi_prices.head()

In [ ]:
normalized_price_frames = []

for symbol in sample_tickers:
    if symbol not in raw_multi_prices.columns.get_level_values(0):
        continue

    symbol_df = raw_multi_prices[symbol].copy()
    symbol_df = symbol_df.reset_index().rename(columns={"Date": "price_date"})
    symbol_df["symbol"] = symbol
    normalized_price_frames.append(symbol_df)

prices_daily = pd.concat(normalized_price_frames, ignore_index=True)
prices_daily.columns = [str(col).strip().lower().replace(" ", "_") for col in prices_daily.columns]

ordered_columns = [
    "symbol",
    "price_date",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
    "dividends",
    "stock_splits",
]

existing_columns = [column for column in ordered_columns if column in prices_daily.columns]
prices_daily = prices_daily[existing_columns]
prices_daily.head(10)

In [ ]:
print("Row count:", len(prices_daily))
print("Distinct symbols:", prices_daily["symbol"].nunique())
print("Date range:", prices_daily["price_date"].min(), "to", prices_daily["price_date"].max())
print("Duplicate business key count:", prices_daily.duplicated(subset=["symbol", "price_date"]).sum())

prices_daily.isna().sum()

## Step 4. Explore corporate actions

Dividends and stock splits are important because they explain differences between raw close and adjusted close. They are also useful datasets on their own.

In [ ]:
aapl_actions = ticker.actions.copy()
aapl_dividends = ticker.dividends.copy()
aapl_splits = ticker.splits.copy()

print("Actions shape:", aapl_actions.shape)
display(aapl_actions.tail(10))

print("Dividends shape:", aapl_dividends.shape)
display(aapl_dividends.tail(10))

print("Splits shape:", aapl_splits.shape)
display(aapl_splits.tail(10))

## Step 5. Explore company metadata

This is useful for later building dimensions such as `dim_company`, `dim_sector`, and dashboard filters.

Important note:
- metadata fields can vary by ticker
- some fields may be missing
- we should land the raw payload in bronze first and only normalize stable fields in silver

In [ ]:
ticker.fast_info

In [ ]:
aapl_info = ticker.info

selected_info_fields = {
    key: aapl_info.get(key)
    for key in [
        "symbol",
        "shortName",
        "longName",
        "quoteType",
        "sector",
        "industry",
        "country",
        "currency",
        "exchange",
        "marketCap",
        "enterpriseValue",
        "fullTimeEmployees",
    ]
}

pd.Series(selected_info_fields)

In [ ]:
metadata_rows = []

for symbol in sample_tickers:
    symbol_ticker = yf.Ticker(symbol)
    info = symbol_ticker.info
    metadata_rows.append(
        {
            "symbol": symbol,
            "short_name": info.get("shortName"),
            "long_name": info.get("longName"),
            "quote_type": info.get("quoteType"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "country": info.get("country"),
            "currency": info.get("currency"),
            "exchange": info.get("exchange"),
            "market_cap": info.get("marketCap"),
        }
    )

company_metadata = pd.DataFrame(metadata_rows)
company_metadata

## Step 6. Bronze layer design

For the bronze layer, keep data as raw as possible.

Recommended raw datasets from Yahoo Finance:
- `market_prices_daily`
- `corporate_actions`
- `company_metadata`

Recommended partition pattern in S3:

`s3://financial-data/bronze/source=yahoo_finance/dataset=market_prices_daily/symbol=AAPL/ingestion_date=2026-03-22/file.json`

Important metadata to store with each bronze record:
- source
- dataset
- symbol
- ingestion timestamp
- request parameters
- raw payload

In [ ]:
ingestion_ts = datetime.now(timezone.utc).isoformat()

bronze_market_prices_payload = {
    "source": "yahoo_finance",
    "dataset": "market_prices_daily",
    "symbol": "AAPL",
    "ingestion_ts_utc": ingestion_ts,
    "request_params": {
        "start": analysis_start_date.strftime("%Y-%m-%d"),
        "end": analysis_end_date.strftime("%Y-%m-%d"),
        "interval": "1d",
        "auto_adjust": False,
        "actions": True,
    },
    "raw_payload": aapl_history.reset_index().to_dict(orient="records"),
}

print(json.dumps(bronze_market_prices_payload, indent=2, default=str)[:2500])

In [ ]:
ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

bronze_relative_path = Path(
    f"bronze/source=yahoo_finance/dataset=market_prices_daily/symbol=AAPL/ingestion_date={ingestion_date}/aapl_prices.json"
)

bronze_relative_path.as_posix()

## Step 7. What we learned from this notebook

Best Yahoo Finance datasets for this project:
- daily OHLCV prices
- dividends and stock splits
- company metadata for dimensions

Best use of Yahoo Finance in your architecture:
- bronze: land raw market and metadata payloads
- silver: normalize prices and metadata into tabular datasets
- gold: combine with Alpha Vantage fundamentals and macro indicators for analytics

Suggested next notebook:
- `02_alpha_vantage_exploration.ipynb`

There we should explore:
- daily adjusted stock prices
- company overview
- income statement
- balance sheet
- FX daily
- one macro indicator such as GDP